In [15]:
import nfl_data_py as nfl
import pandas as pd
from pathlib import Path

# Create PickleFiles directory in Models/ (one level up from this notebook)
Path("../PickleFiles").mkdir(exist_ok=True)

years = list(range(2013, 2024))

# --- Get team points for/against from schedules ---
schedules = nfl.import_schedules(years)
schedules = schedules[(schedules['game_type'] == 'REG') & (schedules['home_score'].notna())].copy()

# Home games
home = schedules.groupby(['season', 'home_team']).agg(
    pts_for=('home_score', 'sum'),
    pts_against=('away_score', 'sum'),
    games=('home_score', 'count')
).reset_index().rename(columns={'home_team': 'team'})

# Away games
away = schedules.groupby(['season', 'away_team']).agg(
    pts_for=('away_score', 'sum'),
    pts_against=('home_score', 'sum'),
    games=('away_score', 'count')
).reset_index().rename(columns={'away_team': 'team'})

team_pts = pd.concat([home, away]).groupby(['season', 'team']).sum().reset_index()
team_pts['offensive_points_per_game'] = team_pts['pts_for'] / team_pts['games']
team_pts['defensive_points_per_game'] = team_pts['pts_against'] / team_pts['games']

# --- Get team rushing/passing yards from weekly data ---
weekly = nfl.import_weekly_data(years, columns=[
    'player_id', 'recent_team', 'season', 'week',
    'rushing_yards', 'passing_yards'
])

team_yards = weekly.groupby(['season', 'recent_team']).agg(
    team_rush_yards=('rushing_yards', 'sum'),
    team_passing_yards=('passing_yards', 'sum')
).reset_index().rename(columns={'recent_team': 'team'})
team_yards['team_total_yards'] = team_yards['team_rush_yards'] + team_yards['team_passing_yards']

# --- Merge into one DataFrame ---
AVgrades = pd.merge(team_pts, team_yards, on=['season', 'team'], how='inner')

# Teams already use NFL abbreviations (ARI, BAL, etc.) from nfl_data_py

AVgrades = AVgrades.rename(columns={
    'pts_for': 'total_offense_points',
    'season': 'year'
})

print(f"AVgrades shape: {AVgrades.shape}")
print(f"Years: {sorted(AVgrades['year'].unique())}")
print(f"Teams per year: {AVgrades.groupby('year')['team'].nunique().unique()}")
AVgrades.head()

Downcasting floats.
AVgrades shape: (338, 10)
Years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Teams per year: [29 30 31 32]


,year,team,total_offense_points,pts_against,games,offensive_points_per_game,defensive_points_per_game,team_rush_yards,team_passing_yards,team_total_yards
0,2013,ARI,379,324,16,23.6875,20.2500,1540.0,4291.0,5831.0
1,2013,ATL,353,443,16,22.0625,27.6875,1247.0,4549.0,5796.0
2,2013,BAL,320,352,16,20.0000,22.0000,1328.0,3914.0,5242.0
3,2013,BUF,339,388,16,21.1875,24.2500,2307.0,3373.0,5680.0
4,2013,CAR,366,241,16,22.8750,15.0625,2119.0,3646.0,5765.0


In [16]:
AVgrades = AVgrades.sort_values(by=['year','team'])

# Using points per game as proxy for points per drive (ratio to league avg is nearly identical)
AVgrades['league_avg_offensive_ppg'] = AVgrades.groupby('year')['offensive_points_per_game'].transform('mean')
AVgrades['league_avg_defensive_ppg'] = AVgrades.groupby('year')['defensive_points_per_game'].transform('mean')

AVgrades["team_offense_points"] = (100 * AVgrades['offensive_points_per_game']) / AVgrades['league_avg_offensive_ppg']
AVgrades["team_points_for_o_line"] = (5/11) * AVgrades["team_offense_points"]
AVgrades["team_points_for_skill_positions"] = AVgrades["team_offense_points"] - AVgrades["team_points_for_o_line"]
AVgrades["team_points_for_rushers"] = AVgrades["team_points_for_skill_positions"] * 0.22 * (AVgrades["team_rush_yards"]/AVgrades["team_total_yards"]) / 0.37
AVgrades["team_points_for_passers"] = (AVgrades["team_points_for_skill_positions"] - AVgrades["team_points_for_rushers"]) * 0.26
AVgrades["team_points_for_receivers"] = (AVgrades["team_points_for_skill_positions"] - AVgrades["team_points_for_rushers"]) * 0.74
AVgrades["M"] = AVgrades["defensive_points_per_game"] / AVgrades['league_avg_defensive_ppg']
AVgrades["team_defense_points"] = 100 * ((1+2*AVgrades["M"]-AVgrades["M"]**2) / (2*AVgrades["M"]))

AVgrades

,year,team,total_offense_points,pts_against,games,offensive_points_per_game,defensive_points_per_game,team_rush_yards,team_passing_yards,team_total_yards,league_avg_offensive_ppg,league_avg_defensive_ppg,team_offense_points,team_points_for_o_line,team_points_for_skill_positions,team_points_for_rushers,team_points_for_passers,team_points_for_receivers,M,team_defense_points
0,2013,ARI,379,324,16,23.687500,20.250000,1540.0,4291.0,5831.0,23.532328,23.318966,100.659401,45.754273,54.905128,8.622070,12.033595,34.249463,0.868392,114.158099
1,2013,ATL,353,443,16,22.062500,27.687500,1247.0,4549.0,5796.0,23.532328,23.318966,93.754007,42.615458,51.138549,6.541953,11.595115,33.001481,1.187338,82.744086
2,2013,BAL,320,352,16,20.000000,22.000000,1328.0,3914.0,5242.0,23.532328,23.318966,84.989468,38.631576,46.357892,6.983059,10.237457,29.137376,0.943438,105.825745
3,2013,BUF,339,388,16,21.187500,24.250000,2307.0,3373.0,5680.0,23.532328,23.318966,90.035718,40.925326,49.110391,11.860243,9.685038,27.565110,1.039926,96.084038
4,2013,CAR,366,241,16,22.875000,15.062500,2119.0,3646.0,5765.0,23.532328,23.318966,97.206704,44.184865,53.021838,11.587971,10.772806,30.661062,0.645933,145.110682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
333,2023,SEA,364,402,17,21.411765,23.647059,1580.0,4167.0,5747.0,21.768382,21.768382,98.361763,44.709892,53.651871,8.770446,11.669170,33.212254,1.086303,91.712525
334,2023,SF,491,298,17,28.882353,17.529412,2765.0,5372.0,8137.0,21.768382,21.768382,132.680290,60.309223,72.371068,14.622335,15.014670,42.734062,0.805269,121.827554
335,2023,TB,348,325,17,20.470588,19.117647,1717.0,4730.0,6447.0,21.768382,21.768382,94.038169,42.744622,51.293547,8.122623,11.224440,31.946484,0.878230,113.021191
336,2023,TEN,305,367,17,17.941176,21.588235,1846.0,3512.0,5358.0,21.768382,21.768382,82.418510,37.462959,44.955551,9.209441,9.293989,26.452122,0.991724,100.831016


In [17]:
AVgrades["team_receiving_yards"] = AVgrades["team_passing_yards"]

In [18]:
AVgrades = AVgrades.rename(columns={'team_points_for_o_line': 'oline', 'team_points_for_rushers': 'rb', 'team_points_for_passers': 'qb', 'team_points_for_receivers':'wrte', 'team_defense_points':'dst', 'year':'season'})
AVgrades.to_pickle("../PickleFiles/AVgrades.pkl")

In [19]:
AVgrades1 = AVgrades[['team','oline','qb','rb','wrte','dst','season']]
AVgrades1.to_pickle("../PickleFiles/AVbyPositionGroup.pkl")